# Settings

Import the libraries, packages and files required

In [1]:
from pathlib import Path
import pandas as pd
import geopandas as gpd
import json

In [2]:
DATA_DIR = Path("data")
GPKG_PATH = DATA_DIR / "snapshot.gpkg"

# 1. Attribute completeness

### Requirements

1. **Settings:** the required attributes list per class.
2. **Check each rule per feature:** one column per rule where True or False values are stored if the rule applies. For example has_name. Values like surface or religion are checked in the tags column, which holds several feature's tags.
3. **Feature score:** the share of rules it passes, from 0 to 1. A road with a name but no surface gets 0.5.
4. **Class score:** the average of its feature scores. This is the S for attribute completeness in the DQI in equation (1).
5. **Save the result:** a completeness table (feature_id, rule columns, score) and the errors (feature_id plus the rules it failed), so there's easy access or way to filter the failing features in QGIS.

### 1.1 Defining the attribute list and the rules

In [ ]:
MAIN_ROADS = ["motorway", "trunk", "primary", "secondary"]   # updated according the meeting 10/11/2026

PLACEHOLDERS = ["road", "fixme", "fix"]   # values that count as missing

RULES = {
    "roads": [
        {"rule": "has_name_or_ref", "tags": ["name", "ref"], "applies_to": {"highway": MAIN_ROADS}, "missing_values": PLACEHOLDERS},
        {"rule": "has_surface", "tags": ["surface"], "applies_to": {"highway": MAIN_ROADS}},
    ],
    "buildings": [
        {"rule": "has_building_type", "tags": ["building"], "applies_to": None, "missing_values": ["yes"]},
    ],
    "pois": [
        {"rule": "has_name", "tags": ["name"], "applies_to": None, "missing_values": PLACEHOLDERS},
        {"rule": "has_religion", "tags": ["religion"], "applies_to": {"amenity": ["place_of_worship"]}},
    ],
}

In [14]:
def check_rule(tags, rule):

    # 1. check if the rule apply to this feature
    conditions = rule["applies_to"]
    if conditions is not None:
        for key, allowed_values in conditions.items():
            feature_value = tags.get(key)
            if feature_value not in allowed_values:
                return None   # rule does not apply

    # 2. values that count as missing (if there are)
    if "missing_values" in rule:
        missing_values = rule["missing_values"]
    else:
        missing_values = []

    # 3. the rule passes if any of its tags has a useful value
    for tag_name in rule["tags"]:
        value = tags.get(tag_name)
        if value is None:
            continue   # tag not present, try the next one

        clean_value = value.strip().lower()   
        if clean_value == "":
            continue   # empty value (or only spaces), try the next one

        if clean_value in missing_values:
            continue   # placeholder, try the next one

        return True    # useful value found

    return False       # no tag had a useful value

#### 1.1.1 Roads

Checking the rules for roads

In [26]:
one_road = {"highway": "primary", "name": "FIX", "ref": "5"}
print(check_rule(one_road, RULES["roads"][0]))  

True


In [ ]:
one_road = {"highway": "primary", "name": "FIX", "ref": "5", "surface":"a"} 
print(check_rule(one_road, RULES["roads"][1]))  

True


Check the rules to all of the roads

In [31]:
roads = gpd.read_file(GPKG_PATH, layer="roads")
tags = roads["tags"].apply(json.loads)

rule_cols = []
for rule in RULES["roads"]:
    roads[rule["rule"]] = pd.array([check_rule(t, rule) for t in tags], dtype="boolean")
    rule_cols.append(rule["rule"])

roads["passed"] = roads[rule_cols].sum(axis=1)
roads["applicable"] = roads[rule_cols].notna().sum(axis=1)
roads["score"] = roads["passed"] / roads["applicable"]
print(roads[rule_cols + ["score"]].describe(include="all"))
print("Class score (roads):", roads["score"].mean())

       has_name_or_ref has_surface     score
count              203         203     203.0
unique               2           2      <NA>
top              False       False      <NA>
freq               117         188      <NA>
mean               NaN         NaN  0.248768
std                NaN         NaN  0.304156
min                NaN         NaN       0.0
25%                NaN         NaN       0.0
50%                NaN         NaN       0.0
75%                NaN         NaN       0.5
max                NaN         NaN       1.0
Class score (roads): 0.24876847290640394


Calculating roads score

In [ ]:
# 1. To get the result according the lenght (m, not degrees), suggested in the meeting 10/11/2026
roads_metric = roads.to_crs(roads.estimate_utm_crs())
roads["length_m"] = roads_metric.length

# 2. keep only roads where at least one rule applies
scored_roads = roads[roads["applicable"] > 0]

# 3. weighted score = sum(score x length) / sum(length)
weighted_sum = 0
total_length = 0
for index, road in scored_roads.iterrows():
    weighted_sum = weighted_sum + road["score"] * road["length_m"]
    total_length = total_length + road["length_m"]

class_score_roads = weighted_sum / total_length
print("Class score (roads, by count):", roads["score"].mean())
print("Class score (roads, by length):", class_score_roads)

Class score (roads, by count): 0.24876847290640394
Class score (roads, by length): 0.17373732984890145


### 1.1.2 Polygons

### 1.1.3 POIs

## 1.2 Class score

## 1.3 Save the result for attribute completeness